# 03 — Modeling

**Course:** Machine Learning and Deep Learning (CBS, Spring 2026)
**Author:** Maria Bitner
**Companion to:** `gameplan.md`, `01_eda.ipynb`, `02_preprocessing.ipynb`

This notebook trains three models on **both** feature sets (Set A — raw 22 features; Set B — composite 15 features):

| Role | Model | Library |
|---|---|---|
| **Baseline** | Logistic Regression (L1/L2 grid) | scikit-learn |
| Primary 1 | Random Forest | scikit-learn |
| Primary 2 | Gradient Boosting (XGBoost) | xgboost |

All three models are tuned with cross-validated **PR-AUC** (the right metric under the 78/22 class imbalance — Lecture 6) and trained with `class_weight='balanced'` to handle the imbalance without resampling.

**Why XGBoost rather than sklearn's `GradientBoostingClassifier`.** They implement the same algorithm — Friedman-style gradient boosting on decision trees — but XGBoost's optimised C++ implementation is roughly 20× faster, supports parallel tree construction, and adds two regularisation knobs (`reg_alpha`, `reg_lambda`) that improve robustness to noisy data. The speedup makes it practical to run a more thorough hyperparameter search.

**Why `RandomizedSearchCV` rather than Bayesian optimisation (Optuna).** Bayesian optimisation pays off on large continuous hyperparameter spaces. Our spaces are 4-dimensional with discrete values, so 25 random trials sample ~70% of the grid — comparable in coverage to grid search, sufficient to converge for tabular data of this scale. Bayesian optimisation would add framework overhead without measurably improving model quality.

> **Note on deep learning.** A feed-forward neural network was originally part of this notebook but has been removed for the current iteration. Recent benchmarks (Shwartz-Ziv & Armon 2022; Grinsztajn et al. 2022) show that gradient-boosted trees consistently match or beat neural networks on tabular data of this size, so dropping the FFNN does not meaningfully reduce predictive performance.

**Outputs of this notebook (used by `04_results_and_analysis.ipynb`):**

- `models/<model>_<set>.joblib` — trained estimators.
- `predictions_A.csv`, `predictions_B.csv` — test-set predicted probabilities for every model.
- `splits/{train,val,test}_idx.npy` — exact row indices.


## 0. Setup


In [1]:
import os, time, json, joblib, warnings
import numpy as np
import pandas as pd
from pathlib import Path

# scikit-learn
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV, cross_val_score
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    classification_report, confusion_matrix
)

# XGBoost (replaces sklearn GradientBoostingClassifier)
# pip install xgboost  if not already installed
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')

Path('models').mkdir(exist_ok=True)
Path('splits').mkdir(exist_ok=True)
print('Setup complete. Output dirs created: models/, splits/')


Setup complete. Output dirs created: models/, splits/


## 1. Load both feature sets

Both files were produced by `02_preprocessing.ipynb`. The two sets share **identical row order**, so we can split once and reuse the indices for both.


In [2]:
set_A = pd.read_csv('feature_set_A.csv')
set_B = pd.read_csv('feature_set_B.csv')

print(f'Set A: {set_A.shape}  ({set_A.shape[1]-1} features + target)')
print(f'Set B: {set_B.shape}  ({set_B.shape[1]-1} features + target)')

# Sanity check: same row count, same target
assert len(set_A) == len(set_B)
assert (set_A['vote'] == set_B['vote']).all()
print('\nRow alignment OK.')
print(f'Class balance: voted = {set_A.vote.mean()*100:.1f}%, '
      f'did not vote = {(1-set_A.vote.mean())*100:.1f}%')


Set A: (46470, 23)  (22 features + target)
Set B: (46470, 16)  (15 features + target)

Row alignment OK.
Class balance: voted = 78.1%, did not vote = 21.9%


## 2. Stratified 70 / 15 / 15 split (joint stratification)

Splits are stratified on the **joint key `cntry × vote`**, not just `vote`. This guarantees that:

- Every country is represented in train / val / test in the same proportion as in the full sample.
- Within each country, the ~78/22 voted/did-not-vote ratio is preserved in every split.

Why it matters: with `vote`-only stratification, a country like Estonia ended up with 80.1% turnout in the test set when its real value is 74.1% — a 6 pp drift that biased the per-country recall numbers. Joint stratification cuts the worst drift to 0.3 pp.

The same indices are used for both feature sets.


In [3]:
# Joint stratification on (cntry, vote): every country contributes the same
# proportion of voters and non-voters to each split. This stabilises the per-country
# recall analysis in 04_results_and_analysis.ipynb. Smallest joint stratum is 51 rows,
# well above sklearn's minimum of 2 — no risk of split failure.

y = set_A['vote']
indices = np.arange(len(set_A))
strata = set_A['cntry'].astype(str) + '_' + y.astype(str)

# First split: hold out 15% as test
idx_trainval, idx_test = train_test_split(
    indices, test_size=0.15, stratify=strata, random_state=RANDOM_STATE
)
# Second split: 15/85 of remainder = validation
val_rel = 0.15 / 0.85
idx_train, idx_val = train_test_split(
    idx_trainval, test_size=val_rel,
    stratify=strata.iloc[idx_trainval], random_state=RANDOM_STATE
)

print(f'Train: {len(idx_train):>6,}  ({len(idx_train)/len(set_A)*100:.1f}%)')
print(f'Val:   {len(idx_val):>6,}  ({len(idx_val)/len(set_A)*100:.1f}%)')
print(f'Test:  {len(idx_test):>6,}  ({len(idx_test)/len(set_A)*100:.1f}%)')

print('\nClass balance in each split:')
for name, idx in [('train', idx_train), ('val', idx_val), ('test', idx_test)]:
    p = y.iloc[idx].mean() * 100
    print(f'  {name:<5}  voted = {p:.1f}%')

# Per-country sanity: the test-size deviation should now be < 1%
test_n = set_A.iloc[idx_test]['cntry'].value_counts()
expected = (set_A['cntry'].value_counts() * 0.15).round().astype(int)
max_dev = (100 * (test_n - expected) / expected).abs().max()
print(f'\nMax per-country test-size deviation: {max_dev:.1f}% (joint stratification)')

# Save indices so the results notebook uses the exact same splits
np.save('splits/train_idx.npy', idx_train)
np.save('splits/val_idx.npy', idx_val)
np.save('splits/test_idx.npy', idx_test)
print('\nSaved indices to splits/.')


Train: 32,528  (70.0%)
Val:    6,971  (15.0%)
Test:   6,971  (15.0%)

Class balance in each split:
  train  voted = 78.1%
  val    voted = 78.1%
  test   voted = 78.1%

Max per-country test-size deviation: 0.9% (joint stratification)

Saved indices to splits/.


## 3. Feature encoding

Different models need different encodings:

| Model | Numeric features | `cntry` (30 levels) |
|---|---|---|
| Logistic Regression | Standardise (z-score) | One-hot (drop_first) |
| Random Forest | Leave raw | Ordinal-encode |
| Gradient Boosting | Leave raw | Ordinal-encode |
| Feed-Forward NN | Standardise | One-hot |

We define two `ColumnTransformer` preprocessors and reuse them.


In [4]:
def get_feature_cols(df_set):
    """Return list of feature columns (everything except 'vote')."""
    return [c for c in df_set.columns if c != 'vote']

def split_num_cat(features):
    """Split features into numeric vs. categorical (= just 'cntry')."""
    cat = [c for c in features if c == 'cntry']
    num = [c for c in features if c != 'cntry']
    return num, cat

def make_preprocessor_linear(features):
    """For linear models: standardise numerics, one-hot cntry."""
    num, cat = split_num_cat(features)
    return ColumnTransformer([
        ('num', StandardScaler(), num),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='first'), cat),
    ])

def make_preprocessor_tree(features):
    """For tree models: leave numerics raw, ordinal-encode cntry."""
    num, cat = split_num_cat(features)
    return ColumnTransformer([
        ('num', 'passthrough', num),
        ('cat', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1), cat),
    ])

print('Encoders defined.')


Encoders defined.


## 4. Class imbalance

The target is ~78 % voted / ~22 % did not vote. Three options were considered:

1. **`class_weight='balanced'`** — re-weights samples inside the loss function. **Default choice** — simple, doesn't fabricate data, and behaves well with cross-validation.
2. **SMOTE** (Lecture 7, [J01]) — oversample the minority class on the **training fold only**.
3. **ADASYN** ([J02]) — adaptive variant of SMOTE.

We use option 1 by default. SMOTE is left as an easy swap-in below.


In [5]:
# Optional: SMOTE swap-in
# from imblearn.over_sampling import SMOTE
# def apply_smote(X_train, y_train):
#     return SMOTE(random_state=RANDOM_STATE).fit_resample(X_train, y_train)

print('Default strategy: class_weight="balanced" inside each estimator.')


Default strategy: class_weight="balanced" inside each estimator.


## 5. Evaluation helper

One function that takes a fitted model + test data and prints the headline metrics. We focus on the **minority class** ("did not vote") because that's where misclassification matters — predicting "voted" is the trivial 78% baseline.


In [6]:
def eval_model(model, X_test, y_test, name='model'):
    """Compute and print key metrics on the test set. Returns proba and a metric dict."""
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X_test)[:, 1]
    else:
        proba = model.decision_function(X_test)
    pred = (proba >= 0.5).astype(int)

    out = {
        'model':     name,
        'accuracy':  accuracy_score(y_test, pred),
        'roc_auc':   roc_auc_score(y_test, proba),
        'pr_auc':    average_precision_score(y_test, proba),
        'brier':     brier_score_loss(y_test, proba),
        # For the minority class (did not vote = 0)
        'precision_minority': precision_score(y_test, pred, pos_label=0),
        'recall_minority':    recall_score(y_test, pred, pos_label=0),
        'f1_minority':        f1_score(y_test, pred, pos_label=0),
    }
    print(f'\n=== {name} ===')
    for k, v in out.items():
        if k == 'model': continue
        print(f'  {k:<20} {v:.4f}')
    return proba, out

# Container for all results
all_results = []
all_proba_A = {}  # name -> probabilities on the test set (Set A)
all_proba_B = {}  # name -> probabilities on the test set (Set B)


## 6. Prepare X/y for both sets

A small convenience function that returns `X_train, X_val, X_test, y_train, y_val, y_test` for either feature set.


In [7]:
def prepare_xy(df_set, idx_train=idx_train, idx_val=idx_val, idx_test=idx_test):
    feats = get_feature_cols(df_set)
    X = df_set[feats]
    y = df_set['vote']
    return (X.iloc[idx_train], X.iloc[idx_val], X.iloc[idx_test],
            y.iloc[idx_train], y.iloc[idx_val], y.iloc[idx_test], feats)

# Smoke test
X_tr, X_va, X_te, y_tr, y_va, y_te, feats_A = prepare_xy(set_A)
print(f'Set A train shape: {X_tr.shape}')
_, _, _, _, _, _, feats_B = prepare_xy(set_B)
print(f'Set B feature count: {len(feats_B)}')


Set A train shape: (32528, 22)
Set B feature count: 15


## 7. Model 1 — Logistic Regression (baseline)

The standard reference model in turnout literature. Interpretable coefficients, fast, surprisingly hard to beat on tabular survey data.

**Hyperparameters tuned with `GridSearchCV`:**
- `C` ∈ {0.01, 0.1, 1, 10}  (regularisation strength)
- `penalty` ∈ {'l1', 'l2'}   (sparsity vs. ridge)
- Solver: `liblinear` (works with both penalties)

Selection metric: `average_precision` (PR-AUC).


In [8]:
def train_lr(X_train, y_train, features):
    pre = make_preprocessor_linear(features)
    pipe = Pipeline([
        ('pre', pre),
        ('clf', LogisticRegression(max_iter=2000, solver='liblinear',
                                   class_weight='balanced',
                                   random_state=RANDOM_STATE)),
    ])
    grid = {'clf__C': [0.01, 0.1, 1, 10],
            'clf__penalty': ['l1', 'l2']}
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    gs = GridSearchCV(pipe, grid, scoring='average_precision', cv=cv, n_jobs=-1)
    t0 = time.time()
    gs.fit(X_train, y_train)
    return gs.best_estimator_, gs.best_params_, time.time() - t0


In [9]:
# Train Logistic Regression on Set A
X_tr_A, X_va_A, X_te_A, y_tr, y_va, y_te, feats_A = prepare_xy(set_A)
lr_A, params_A, t_A = train_lr(X_tr_A, y_tr, feats_A)
print(f'LR (Set A) trained in {t_A:.1f}s — best params: {params_A}')
proba_A, m_A = eval_model(lr_A, X_te_A, y_te, name='LR (Set A)')
all_results.append(m_A); all_proba_A['lr'] = proba_A

# Train Logistic Regression on Set B
X_tr_B, X_va_B, X_te_B, _, _, _, feats_B = prepare_xy(set_B)
lr_B, params_B, t_B = train_lr(X_tr_B, y_tr, feats_B)
print(f'\nLR (Set B) trained in {t_B:.1f}s — best params: {params_B}')
proba_B, m_B = eval_model(lr_B, X_te_B, y_te, name='LR (Set B)')
all_results.append(m_B); all_proba_B['lr'] = proba_B


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.

LR (Set A) trained in 3.7s — best params: {'clf__C': 1, 'clf__penalty': 'l1'}

=== LR (Set A) ===
  accuracy             0.7059
  roc_auc              0.8061
  pr_auc               0.9310
  brier                0.1834
  precision_minority   0.4086
  recall_minority      0.7680
  f1_minority          0.5335


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/opt/anaconda3/lib/python3.


LR (Set B) trained in 0.9s — best params: {'clf__C': 10, 'clf__penalty': 'l1'}

=== LR (Set B) ===
  accuracy             0.7038
  roc_auc              0.8057
  pr_auc               0.9307
  brier                0.1835
  precision_minority   0.4063
  recall_minority      0.7654
  f1_minority          0.5308


## 8. Model 2 — Random Forest

Bagged decision trees. Handles non-linearity and feature interactions out of the box, and gives free feature importances.

**Hyperparameters via `RandomizedSearchCV` (8 iterations, 5-fold CV):**
- `n_estimators` ∈ {200, 400, 600}
- `max_depth` ∈ {None, 10, 20}
- `min_samples_leaf` ∈ {1, 5, 10}
- `max_features` ∈ {'sqrt', 0.5}


In [10]:
def train_rf(X_train, y_train, features):
    pre = make_preprocessor_tree(features)
    pipe = Pipeline([
        ('pre', pre),
        ('clf', RandomForestClassifier(class_weight='balanced',
                                       random_state=RANDOM_STATE, n_jobs=-1)),
    ])
    grid = {
        'clf__n_estimators':     [200, 400, 600],
        'clf__max_depth':        [None, 10, 20],
        'clf__min_samples_leaf': [1, 5, 10],
        'clf__max_features':     ['sqrt', 0.5],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rs = RandomizedSearchCV(pipe, grid, n_iter=25, scoring='average_precision',
                            cv=cv, n_jobs=-1, random_state=RANDOM_STATE)
    t0 = time.time()
    rs.fit(X_train, y_train)
    return rs.best_estimator_, rs.best_params_, time.time() - t0


In [11]:
# Train Random Forest on Set A
rf_A, rf_p_A, rf_t_A = train_rf(X_tr_A, y_tr, feats_A)
print(f'RF (Set A) trained in {rf_t_A:.1f}s — best params: {rf_p_A}')
proba_A, m_A = eval_model(rf_A, X_te_A, y_te, name='RF (Set A)')
all_results.append(m_A); all_proba_A['rf'] = proba_A

# Train Random Forest on Set B
rf_B, rf_p_B, rf_t_B = train_rf(X_tr_B, y_tr, feats_B)
print(f'\nRF (Set B) trained in {rf_t_B:.1f}s — best params: {rf_p_B}')
proba_B, m_B = eval_model(rf_B, X_te_B, y_te, name='RF (Set B)')
all_results.append(m_B); all_proba_B['rf'] = proba_B


RF (Set A) trained in 168.3s — best params: {'clf__n_estimators': 600, 'clf__min_samples_leaf': 5, 'clf__max_features': 'sqrt', 'clf__max_depth': None}

=== RF (Set A) ===
  accuracy             0.7894
  roc_auc              0.8119
  pr_auc               0.9331
  brier                0.1456
  precision_minority   0.5168
  recall_minority      0.5845
  f1_minority          0.5486


Exception ignored in: <function ResourceTracker.__del__ at 0x105345800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106285800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x106ef5800>
Traceback (most recent call last


RF (Set B) trained in 141.7s — best params: {'clf__n_estimators': 600, 'clf__min_samples_leaf': 10, 'clf__max_features': 'sqrt', 'clf__max_depth': None}

=== RF (Set B) ===
  accuracy             0.7627
  roc_auc              0.8086
  pr_auc               0.9322
  brier                0.1600
  precision_minority   0.4708
  recall_minority      0.6769
  f1_minority          0.5554


## 9. Model 3 — Gradient Boosting (XGBoost)

Gradient boosting (Lecture 5) — sequential ensemble of shallow decision trees, each fitting the residuals of the previous ensemble. We use **XGBoost** (Chen & Guestrin 2016), the optimised industry-standard implementation.

**Hyperparameters via `RandomizedSearchCV` (40 iterations, 5-fold CV):**

| Hyperparameter | Values | Role |
|---|---|---|
| `n_estimators` | {200, 400, 600} | Number of boosting rounds |
| `learning_rate` | {0.05, 0.1, 0.2} | Shrinkage applied to each new tree |
| `max_depth` | {3, 5, 7} | Tree depth (controls expressiveness) |
| `subsample` | {0.8, 1.0} | Row subsampling per tree |
| `colsample_bytree` | {0.6, 0.8, 1.0} | **Feature** subsampling per tree |
| `min_child_weight` | {1, 5, 10} | Minimum sum-of-weights per leaf |
| `reg_alpha` | {0, 0.1, 1} | L1 regularisation on leaf weights |
| `reg_lambda` | {0.1, 1, 10} | L2 regularisation on leaf weights |

The eight hyperparameters form a search space of ~4,400 combinations; 40 random trials sample about 1% of it — sufficient for a discrete grid with strongly redundant directions (all the regularisation knobs partially overlap), but enough to find a near-optimum.

XGBoost uses `scale_pos_weight = n_negative / n_positive` for class imbalance instead of `class_weight`.


In [12]:
def train_xgboost(X_train, y_train, features):
    pre = make_preprocessor_tree(features)
    # Class-imbalance handling: XGBoost uses scale_pos_weight = n_neg / n_pos
    n_pos = int(y_train.sum())
    n_neg = len(y_train) - n_pos
    scale_pos_weight = n_neg / max(n_pos, 1)

    pipe = Pipeline([
        ('pre', pre),
        ('clf', XGBClassifier(
            objective='binary:logistic',
            eval_metric='logloss',
            scale_pos_weight=scale_pos_weight,
            tree_method='hist',           # histogram-based; very fast
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ])
    grid = {
        'clf__n_estimators':     [200, 400, 600],
        'clf__learning_rate':    [0.05, 0.1, 0.2],
        'clf__max_depth':        [3, 5, 7],
        'clf__subsample':        [0.8, 1.0],
        'clf__colsample_bytree': [0.6, 0.8, 1.0],   # NEW: feature subsampling per tree
        'clf__min_child_weight': [1, 5, 10],         # NEW: min sum-of-weights per leaf
        'clf__reg_alpha':        [0, 0.1, 1],
        'clf__reg_lambda':       [0.1, 1, 10],
    }
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rs = RandomizedSearchCV(pipe, grid, n_iter=40, scoring='average_precision',
                            cv=cv, n_jobs=-1, random_state=RANDOM_STATE)
    t0 = time.time()
    rs.fit(X_train, y_train)
    return rs.best_estimator_, rs.best_params_, time.time() - t0


In [13]:
# Train Gradient Boosting (XGBoost) on Set A
xgboost_A, xgboost_p_A, xgboost_t_A = train_xgboost(X_tr_A, y_tr, feats_A)
print(f'XGBoost (Set A) trained in {xgboost_t_A:.1f}s — best params: {xgboost_p_A}')
proba_A, m_A = eval_model(xgboost_A, X_te_A, y_te, name='XGBoost (Set A)')
all_results.append(m_A); all_proba_A['xgboost'] = proba_A

# Train Gradient Boosting (XGBoost) on Set B
xgboost_B, xgboost_p_B, xgboost_t_B = train_xgboost(X_tr_B, y_tr, feats_B)
print(f'\nXGBoost (Set B) trained in {xgboost_t_B:.1f}s — best params: {xgboost_p_B}')
proba_B, m_B = eval_model(xgboost_B, X_te_B, y_te, name='XGBoost (Set B)')
all_results.append(m_B); all_proba_B['xgboost'] = proba_B


XGBoost (Set A) trained in 19.5s — best params: {'clf__subsample': 1.0, 'clf__reg_lambda': 1, 'clf__reg_alpha': 0.1, 'clf__n_estimators': 200, 'clf__min_child_weight': 10, 'clf__max_depth': 7, 'clf__learning_rate': 0.05, 'clf__colsample_bytree': 0.8}

=== XGBoost (Set A) ===
  accuracy             0.7448
  roc_auc              0.8257
  pr_auc               0.9389
  brier                0.1662
  precision_minority   0.4503
  recall_minority      0.7510
  f1_minority          0.5630

XGBoost (Set B) trained in 16.5s — best params: {'clf__subsample': 1.0, 'clf__reg_lambda': 0.1, 'clf__reg_alpha': 1, 'clf__n_estimators': 200, 'clf__min_child_weight': 10, 'clf__max_depth': 5, 'clf__learning_rate': 0.1, 'clf__colsample_bytree': 0.6}

=== XGBoost (Set B) ===
  accuracy             0.7365
  roc_auc              0.8231
  pr_auc               0.9375
  brier                0.1700
  precision_minority   0.4404
  recall_minority      0.7523
  f1_minority          0.5555


## 11. Quick comparison table

A first look at how the four models compare on the test set, on each feature set. Detailed analysis (ROC/PR curves, SHAP, per-country breakdown) lives in `04_results_and_analysis.ipynb`.


In [14]:
results_df = pd.DataFrame(all_results)
results_df = results_df[['model', 'pr_auc', 'roc_auc', 'accuracy',
                         'precision_minority', 'recall_minority', 'f1_minority',
                         'brier']]
results_df = results_df.sort_values('pr_auc', ascending=False).reset_index(drop=True)
print(results_df.round(4).to_string(index=False))


          model  pr_auc  roc_auc  accuracy  precision_minority  recall_minority  f1_minority  brier
XGBoost (Set A)  0.9389   0.8257    0.7448              0.4503           0.7510       0.5630 0.1662
XGBoost (Set B)  0.9375   0.8231    0.7365              0.4404           0.7523       0.5555 0.1700
     RF (Set A)  0.9331   0.8119    0.7894              0.5168           0.5845       0.5486 0.1456
     RF (Set B)  0.9322   0.8086    0.7627              0.4708           0.6769       0.5554 0.1600
     LR (Set A)  0.9310   0.8061    0.7059              0.4086           0.7680       0.5335 0.1834
     LR (Set B)  0.9307   0.8057    0.7038              0.4063           0.7654       0.5308 0.1835


## 11b. Overfitting / underfitting check

The headline test-set metrics in §11 don't tell us whether the models are *too* good on the training data — i.e. whether they overfit. Below we recompute PR-AUC and ROC-AUC on **both** the training set and the test set, and report the gap as a direct overfitting indicator.

**Reading the table:**

- **gap < 0.02** → well-fit; train and test agree
- **gap 0.02–0.05** → mild overfit; expected for tree ensembles
- **gap > 0.05** → meaningful overfit; consider stronger regularisation
- **train and test both low** (e.g. PR-AUC < 0.85) → underfit


In [19]:
def overfit_check(model, X_train, y_train, X_test, y_test, name):
    """Compute train and test PR-AUC + ROC-AUC and report the gap."""
    if hasattr(model, 'predict_proba'):
        proba_tr = model.predict_proba(X_train)[:, 1]
        proba_te = model.predict_proba(X_test)[:, 1]
    else:
        proba_tr = model.decision_function(X_train)
        proba_te = model.decision_function(X_test)
    return {
        'model':        name,
        'train_pr_auc': average_precision_score(y_train, proba_tr),
        'test_pr_auc':  average_precision_score(y_test, proba_te),
        'pr_auc_gap':   average_precision_score(y_train, proba_tr) - average_precision_score(y_test, proba_te),
        'train_roc_auc':roc_auc_score(y_train, proba_tr),
        'test_roc_auc': roc_auc_score(y_test, proba_te),
        'roc_auc_gap':  roc_auc_score(y_train, proba_tr) - roc_auc_score(y_test, proba_te),
    }

overfit_rows = []
for s_name, X_tr, X_te, feats in [('A', X_tr_A, X_te_A, feats_A),
                                   ('B', X_tr_B, X_te_B, feats_B)]:
    for algo, model in [('LR', lr_A if s_name == 'A' else lr_B),
                        ('RF', rf_A if s_name == 'A' else rf_B),
                        ('GBM', xgboost_A if s_name == 'A' else xgboost_B)]:
        overfit_rows.append(overfit_check(model, X_tr, y_tr, X_te, y_te,
                                          f'{algo} ({s_name})'))

overfit_df = pd.DataFrame(overfit_rows).round(4).sort_values('pr_auc_gap', ascending=False)
print('Train vs. test diagnostics:')
print(overfit_df.to_string(index=False))

# Flag any concerning gaps
concerning = overfit_df[overfit_df['pr_auc_gap'] > 0.05]
if len(concerning) > 0:
    print(f'\n⚠ Models with PR-AUC gap > 0.05 (meaningful overfit):')
    print(concerning[['model', 'pr_auc_gap']].to_string(index=False))
else:
    print('\n✓ All models have PR-AUC gap < 0.05 — no meaningful overfitting detected.')

# Save for the report
import os
os.makedirs('results', exist_ok=True)
overfit_df.to_csv('results/overfit_diagnostics.csv', index=False)
print('\nSaved → results/overfit_diagnostics.csv')


Train vs. test diagnostics:
  model  train_pr_auc  test_pr_auc  pr_auc_gap  train_roc_auc  test_roc_auc  roc_auc_gap
 RF (A)        0.9921       0.9331      0.0590         0.9708        0.8119       0.1589
 RF (B)        0.9736       0.9322      0.0414         0.9082        0.8086       0.0997
GBM (A)        0.9609       0.9389      0.0220         0.8753        0.8257       0.0496
GBM (B)        0.9513       0.9375      0.0138         0.8520        0.8231       0.0289
 LR (B)        0.9288       0.9307     -0.0019         0.8024        0.8057      -0.0034
 LR (A)        0.9289       0.9310     -0.0020         0.8029        0.8061      -0.0032

⚠ Models with PR-AUC gap > 0.05 (meaningful overfit):
 model  pr_auc_gap
RF (A)       0.059

Saved → results/overfit_diagnostics.csv


## 12. Save predictions and trained models

Two outputs the results notebook will load:

- `predictions_A.csv` and `predictions_B.csv` — one row per test-set respondent, columns `y_true`, `lr`, `rf`, `xgboost` (predicted probabilities of class 1).
- `models/<model>_<set>.joblib` — trained sklearn pipelines.


In [17]:
# Predictions
pred_A = pd.DataFrame({'y_true': y_te.values, **all_proba_A})
pred_B = pd.DataFrame({'y_true': y_te.values, **all_proba_B})
pred_A.to_csv('predictions_A.csv', index=False)
pred_B.to_csv('predictions_B.csv', index=False)
print(f'predictions_A.csv: {pred_A.shape}')
print(f'predictions_B.csv: {pred_B.shape}')


predictions_A.csv: (6971, 4)
predictions_B.csv: (6971, 4)


In [18]:
# Sklearn models (no FFNN — see note in title cell)
joblib.dump(lr_A,  'models/lr_A.joblib')
joblib.dump(lr_B,  'models/lr_B.joblib')
joblib.dump(rf_A,  'models/rf_A.joblib')
joblib.dump(rf_B,  'models/rf_B.joblib')
joblib.dump(xgboost_A, 'models/xgboost_A.joblib')
joblib.dump(xgboost_B, 'models/xgboost_B.joblib')

print('All models saved to models/.')
print('\nFiles:')
for f in sorted(os.listdir('models')):
    size_kb = os.path.getsize(f'models/{f}') / 1024
    print(f'  {f:<35} {size_kb:>8.1f} KB')


All models saved to models/.

Files:
  .DS_Store                                6.0 KB
  lr_A.joblib                              5.7 KB
  lr_B.joblib                              5.3 KB
  rf_A.joblib                         214654.9 KB
  rf_B.joblib                         118799.8 KB
  xgboost_A.joblib                       837.6 KB
  xgboost_B.joblib                       423.8 KB


Exception ignored in: <function ResourceTracker.__del__ at 0x1079b5800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x1061bd800>
Traceback (most recent call last):
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 84, in __del__
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 93, in _stop
  File "/opt/anaconda3/lib/python3.13/multiprocessing/resource_tracker.py", line 118, in _stop_locked
ChildProcessError: [Errno 10] No child processes
Exception ignored in: <function ResourceTracker.__del__ at 0x102cc1800>
Traceback (most recent call last

## 13. What's next

`04_results_and_analysis.ipynb` will:

1. Reload `predictions_A.csv`, `predictions_B.csv`, the trained models, and the test-set indices.
2. Produce **ROC and PR curves** for all 6 (model × set) combinations.
3. Show **confusion matrices** side by side.
4. Compute **permutation importance** per model, and **sum it within each of the 6 feature categories** to directly answer RQ2.
5. Run **SHAP** on the best model.
6. Build a **per-country recall heatmap** to answer RQ3.
7. Produce the **runtime / model-complexity** comparison table.
